In [1]:
import polars as pl
import pandas as pd
from datetime import datetime

def audit_parquet_data(file_path: str):
    print(f"🔍 Auditing: {file_path}")
    
    # 1. Load the metadata and first/last rows
    df = pl.read_parquet(file_path)
    
    # 2. Check for required columns
    cols = df.columns
    print(f"✅ Columns found: {cols[:10]}... (Total: {len(cols)})")
    
    # 3. Check Timestamp Scale
    if "time_ns" in cols:
        sample_ts = df["time_ns"][0]
        ts_str = str(abs(int(sample_ts)))
        print(f"📏 Timestamp Length: {len(ts_str)} digits")
        if len(ts_str) == 19:
            print("   -> Scale: Nanoseconds (Correct)")
        elif len(ts_str) == 13:
            print("   -> Scale: Milliseconds (WRONG for current logic)")
        elif len(ts_str) == 10:
            print("   -> Scale: Seconds (WRONG for current logic)")
            
    # 4. Check Sorting (The "Backwards" problem)
    time_col = "time" if "time" in cols else "time_ns"
    first_time = df[time_col][0]
    last_time = df[time_col][-1]
    
    # Convert to human readable if it's raw ns
    if isinstance(first_time, int):
        first_dt = pd.to_datetime(first_time, unit='ns')
        last_dt = pd.to_datetime(last_time, unit='ns')
    else:
        first_dt = first_time
        last_dt = last_time

    print(f"📅 Temporal Range:")
    print(f"   - Start: {first_dt}")
    print(f"   - End:   {last_dt}")

    if first_dt > last_dt:
        print("❌ ERROR: Data is sorted DESCENDING (Newest to Oldest).")
        print("   This causes np.searchsorted to return index 0 for all eras.")
    else:
        print("✅ Data is sorted ASCENDING (Oldest to Newest).")

    # 5. Check for Gaps
    if "time" in cols:
        diffs = df["time"].diff().dt.total_minutes().drop_nulls()
        median_gap = diffs.median()
        max_gap = diffs.max()
        print(f"⏱️ Interval Check:")
        print(f"   - Expected Interval: ~{median_gap} minutes")
        if max_gap > median_gap * 2:
            print(f"   ⚠️ WARNING: Large gap detected! Max gap is {max_gap} minutes.")

    # 6. Null Check for features
    null_counts = df.null_count()
    print("🧹 Null Value Check:")
    for col in ["pct_d_slow", "breakout_1h", "close"]:
        if col in cols:
            nc = null_counts[col][0]
            print(f"   - {col}: {nc} nulls ({(nc/len(df))*100:.2f}%)")

if __name__ == "__main__":
    PATH = r"C:\Users\Owner\airflow-trading\data_lake\base_data_full\base_data_full.parquet"
    audit_parquet_data(PATH)

🔍 Auditing: C:\Users\Owner\airflow-trading\data_lake\base_data_full\base_data_full.parquet
✅ Columns found: ['pair', 'time', 'time_ns', 'open', 'high', 'low', 'close', 'volume', 'era_int', 'market_type']... (Total: 14)
📏 Timestamp Length: 19 digits
   -> Scale: Nanoseconds (Correct)
📅 Temporal Range:
   - Start: 2025-03-09 05:25:00+00:00
   - End:   2024-09-30 04:25:00+00:00
❌ ERROR: Data is sorted DESCENDING (Newest to Oldest).
   This causes np.searchsorted to return index 0 for all eras.


AttributeError: 'DateTimeNameSpace' object has no attribute 'total_minutes'